 Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

PositionalEncoding class

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)  
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0)]

 TransformerDecoderLayer class

In [3]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_ff, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        tgt2 = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask)[0]
        tgt = self.norm1(tgt + self.dropout1(tgt2))

        tgt2 = self.multihead_attn(tgt, memory, memory, attn_mask=memory_mask)[0]
        tgt = self.norm2(tgt + self.dropout2(tgt2))

        tgt2 = self.linear2(F.relu(self.linear1(tgt)))
        tgt = self.norm3(tgt + self.dropout3(tgt2))
        return tgt

TransformerDecoder class

In [4]:
class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, nhead, dim_ff, vocab_size, dropout=0.1, max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_ff, dropout)
            for _ in range(num_layers)
        ])
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        tgt = self.embedding(tgt)
        tgt = self.pos_encoder(tgt)
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, memory_mask)
        return self.output_layer(tgt)

Hyperparameter setup

In [5]:
d_model = 128
nhead = 4
dim_ff = 512
num_layers = 2
vocab_size = 5000
seq_len = 10
batch_size = 4

Model instantiation

In [6]:
decoder = TransformerDecoder(num_layers, d_model, nhead, dim_ff, vocab_size)

Sample input:

In [7]:
tgt = torch.randint(0, vocab_size, (seq_len, batch_size))  
memory = torch.rand(seq_len, batch_size, d_model)

Forward pass:

In [8]:
out = decoder(tgt, memory)
print(out.shape)  
print(out)

torch.Size([10, 4, 5000])
tensor([[[-0.9144,  0.1567, -0.7441,  ...,  0.5872,  0.3035, -0.3402],
         [ 0.2447,  0.3439, -0.0579,  ...,  0.5095, -0.6374, -0.4302],
         [ 0.5072,  0.5377, -0.0102,  ..., -0.2429, -0.0558, -0.4889],
         [-0.0023,  0.2327, -1.3023,  ..., -0.4260, -0.7923,  0.0024]],

        [[-0.0592,  0.1469, -0.3613,  ...,  0.7917,  0.5471, -0.3575],
         [ 0.5612,  0.3425, -0.5751,  ..., -0.2804,  0.5203,  0.7867],
         [ 0.6029,  0.2479, -0.7896,  ...,  0.4707,  0.1677, -0.0445],
         [ 0.4377,  0.1419, -0.8901,  ..., -0.3262,  0.2782, -0.1804]],

        [[ 0.6436, -0.7083, -0.1526,  ...,  0.1403, -0.2979, -0.7241],
         [-0.8183,  0.9886,  0.3815,  ...,  0.0224, -0.5616, -0.5172],
         [ 0.5411,  0.3109, -0.5877,  ..., -0.5683,  0.3951, -0.0025],
         [ 0.1753,  0.5860, -0.9659,  ...,  0.1453,  0.0555, -0.7448]],

        ...,

        [[ 0.1410,  0.5747, -0.7455,  ..., -0.3411, -1.0279,  0.0790],
         [-0.3666,  0.1872,  0.